In [1]:
import os
import sys

sys.path.insert(1, os.path.join(os.getcwd()  , 'Modules/'))

print(sys.path)

import pandas as pd
import numpy as np

import re
import logging
from Modules.Loader_wrangler import *
from Modules.Transformations import *
from Modules.ToTensor import *
from Modules.TravNet import *
from Modules.TravNetUser import TravNet
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
import matplotlib.pyplot as plt
import Modules.config as cfg

['/home/trapfishscott/miniconda3/envs/TravNet/lib/python312.zip', '/home/trapfishscott/Cambridge24.25/D200_ML_econ/ProblemSets/TravNet/Modules/', '/home/trapfishscott/miniconda3/envs/TravNet/lib/python3.12', '/home/trapfishscott/miniconda3/envs/TravNet/lib/python3.12/lib-dynload', '', '/home/trapfishscott/miniconda3/envs/TravNet/lib/python3.12/site-packages']
Training running on: cuda
Training running on: cuda


In [2]:
# Configure basic logging
logging.basicConfig(level=logging.INFO, force=True, format='%(levelname)s: %(message)s')

# Loading, Reading & Transforming to Tensors

In [3]:
# Define data range

years_to_extract = list(range(2017,2018))


Unhash below if you want to load, with `return_raw=True` we can return the orginal merged data frame. If we set `return_raw=False` Then we subset the data frame for the variables used in training the RNN, they can be found and modified in `config.py`

In [4]:
# raw_df = loader(output_file_name="merged_df2017.pkl", chunksize=100000, sample_size=100000, survey_years=years_to_extract, return_raw=True)


In [5]:
# nts_df = loader(output_file_name="merged_df2017.pkl", chunksize=100000, sample_size=100000, survey_years=years_to_extract, return_raw=False)

Otherwise just read in the saved pickle...

In [6]:
nts_df = pd.read_pickle(data_folder +"/merged_df2017.pkl")


### Showcase: At random selects an individuals and shows the transformation process and the final features w/ indices

In [11]:
target_cols = prepare_data_for_LSTM(long_df=nts_df, 
                                    features=cfg.features,
                                    outcomes=cfg.outcomes,
                                    categorical_outcome_vars=cfg.categorical_outcome_vars,
                                    features_numerical=cfg.features_numerical,
                                    features_cyclical=cfg.features_cyclical,
                                    features_one_hot=cfg.features_one_hot,
                                    extra_vars=cfg.extra_vars,
                                    cyclical_encoder=apply_cyclical_encoding,
                                    custom_numerical_scaler=custom_numerical_scaler,
                                    log_transformer=log_transformer,
                                    debug=True,
                                    max_journey_seq=10, 
                                    seq_length = 7,
                                    scikit_minmax_scaler=cfg.standard_mms,
                                    scikit_onehot_scaler=cfg.ohe)

0: TripStart_1
1: TripEnd_1
2: TripStart_2
3: TripEnd_2
4: TripStart_3
5: TripEnd_3
6: TripStart_4
7: TripEnd_4
8: TripStart_5
9: TripEnd_5
10: TripStart_6
11: TripEnd_6
12: TripStart_7
13: TripEnd_7
14: TripStart_8
15: TripEnd_8
16: TripStart_9
17: TripEnd_9
18: TripStart_10
19: TripEnd_10
20: TripDisExSW_1
21: TripDisExSW_2
22: TripDisExSW_3
23: TripDisExSW_4
24: TripDisExSW_5
25: TripDisExSW_6
26: TripDisExSW_7
27: TripDisExSW_8
28: TripDisExSW_9
29: TripDisExSW_10
30: TripPurpose_B01ID_1
31: TripPurpose_B01ID_2
32: TripPurpose_B01ID_3
33: TripPurpose_B01ID_4
34: TripPurpose_B01ID_5
35: TripPurpose_B01ID_6
36: TripPurpose_B01ID_7
37: TripPurpose_B01ID_8
38: TripPurpose_B01ID_9
39: TripPurpose_B01ID_10
40: IsTrip_1
41: IsTrip_2
42: IsTrip_3
43: IsTrip_4
44: IsTrip_5
45: IsTrip_6
46: IsTrip_7
47: IsTrip_8
48: IsTrip_9
49: IsTrip_10
50: DrivLic_B02ID
51: VehAnMileage
52: HHoldEmploy_B01ID
53: VehComMile_B01ID
54: EcoStat_B02ID
55: HHoldNumPeople
56: WkPlace_B01ID
57: Age_B01ID
58: HHol

,TripStart_1,TripEnd_1,TripStart_2,TripEnd_2,TripStart_3,TripEnd_3,TripStart_4,TripEnd_4,TripStart_5,TripEnd_5,...,PSUGOR_B02ID_0.0,PSUGOR_B02ID_1.0,PSUGOR_B02ID_2.0,PSUGOR_B02ID_3.0,PSUGOR_B02ID_4.0,PSUGOR_B02ID_5.0,PSUGOR_B02ID_6.0,PSUGOR_B02ID_7.0,PSUGOR_B02ID_8.0,PSUGOR_B02ID_9.0
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.395833,0.402778,0.416667,0.423611,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.385417,0.392361,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.562500,0.569444,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.409722,0.423611,0.440972,0.458333,0.458333,0.475694,0.555556,0.572917,0.583333,0.590278,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,TripStart_1,TripEnd_1,TripStart_2,TripEnd_2,TripStart_3,TripEnd_3,TripStart_4,TripEnd_4,TripStart_5,TripEnd_5,...,TripDisExSW_1,TripDisExSW_2,TripDisExSW_3,TripDisExSW_4,TripDisExSW_5,TripDisExSW_6,TripDisExSW_7,TripDisExSW_8,TripDisExSW_9,TripDisExSW_10
7,0.395833,0.402778,0.416667,0.423611,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.386294,1.386294,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0
8,0.385417,0.392361,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.562500,0.569444,...,1.609438,0.000000,0.000000,0.000000,0.693147,1.609438,1.609438,0,0,0
9,0.409722,0.423611,0.440972,0.458333,0.458333,0.475694,0.555556,0.572917,0.583333,0.590278,...,1.791759,1.609438,2.302585,1.945910,1.098612,1.098612,1.945910,0,0,0
10,0.388889,0.395833,0.406250,0.413194,0.437500,0.451389,0.479167,0.486111,0.601389,0.604861,...,1.386294,1.386294,1.945910,1.945910,0.405465,0.405465,0.000000,0,0,0
11,0.395833,0.406250,0.423611,0.434028,0.572917,0.583333,0.635417,0.645833,0.000000,0.000000,...,1.945910,1.945910,1.945910,1.945910,0.000000,0.000000,0.000000,0,0,0
12,0.423611,0.437500,0.451389,0.465278,0.781250,0.791667,0.892361,0.902778,0.000000,0.000000,...,1.791759,1.791759,1.945910,1.945910,0.000000,0.000000,0.000000,0,0,0
13,0.395833,0.402778,0.416667,0.430556,0.576389,0.579861,0.595139,0.597917,0.000000,0.000000,...,1.945910,1.945910,0.405465,0.405465,0.000000,0.000000,0.000000,0,0,0


,TripPurpose_B01ID_1,TripPurpose_B01ID_2,TripPurpose_B01ID_3,TripPurpose_B01ID_4,TripPurpose_B01ID_5,TripPurpose_B01ID_6,TripPurpose_B01ID_7,TripPurpose_B01ID_8,TripPurpose_B01ID_9,TripPurpose_B01ID_10,IsTrip_1,IsTrip_2,IsTrip_3,IsTrip_4,IsTrip_5,IsTrip_6,IsTrip_7,IsTrip_8,IsTrip_9,IsTrip_10
7,16.0,16.0,0.0,0.0,0.0,0.0,0.0,0,0,0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0,0
8,16.0,0.0,0.0,0.0,5.0,5.0,5.0,0,0,0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0,0,0
9,16.0,10.0,10.0,10.0,16.0,10.0,10.0,0,0,0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0,0,0
10,16.0,16.0,9.0,9.0,5.0,5.0,0.0,0,0,0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0,0,0
11,16.0,16.0,10.0,10.0,0.0,0.0,0.0,0,0,0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0,0,0
12,16.0,16.0,11.0,11.0,0.0,0.0,0.0,0,0,0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0,0,0
13,16.0,16.0,5.0,5.0,0.0,0.0,0.0,0,0,0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0,0,0


# Showcasing TravNet

In [12]:
tn = TravNet()

INFO: Running on: cuda


### Generating Travel 

*This will be done on 1000 users here to speed things up. For the paper it was done on 10,000 users*

In [13]:
tn.generate_travel_data(1000)

INFO: Shape of X_eval: torch.Size([1000, 14, 1, 78])
/home/trapfishscott/Cambridge24.25/D200_ML_econ/ProblemSets/TravNet/Modules/TravNet.py:204: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experi

### Printing Aggregate Statistics

In [14]:
tn.output_aggregate_stats()

Purpouse Value counts (Gen)
Purpouse        | Proportion
------------------------------
1.0             |       0.61
5.0             |       0.17
10.0            |       0.15
23.0            |       0.07
Purpouse Value counts (True)
Purpouse        | Proportion
------------------------------
1.0             |       0.20
5.0             |       0.11
6.0             |       0.10
10.0            |       0.10
21.0            |       0.07
9.0             |       0.06
23.0            |       0.06
13.0            |       0.06
2.0             |       0.05
16.0            |       0.04
22.0            |       0.03
11.0            |       0.03
7.0             |       0.02
19.0            |       0.02
3.0             |       0.02
14.0            |       0.01
12.0            |       0.01
15.0            |       0.01
4.0             |       0.00
20.0            |       0.00
18.0            |       0.00
8.0             |       0.00
Overall Aggregate stats
Generated
         DoW  TripNum  TripStart  D

### Plotting Histgrams

In [ ]:
tn.create_histograms()